In [15]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense,Flatten,Dropout,Conv2D,MaxPool2D
import matplotlib.pyplot as plt
from tensorflow.keras.utils import load_img,img_to_array

In [16]:
train_dir = r"C:\XPrathmesh\College Stuff\DeepLearning\LA3_Dataset\New Plant Diseases Dataset(Augmented)\train"
val_dir = r"C:\XPrathmesh\College Stuff\DeepLearning\LA3_Dataset\New Plant Diseases Dataset(Augmented)\valid"
image_size = (128,128)
batch_size = 32

In [17]:
train_datagen = ImageDataGenerator(rescale=1.0/255)
val_datagen = ImageDataGenerator(rescale=1.0/255)

In [18]:
train_data = train_datagen.flow_from_directory(train_dir,target_size=image_size,batch_size=batch_size,class_mode='categorical')
val_dir = val_datagen.flow_from_directory(val_dir,target_size=image_size,batch_size=batch_size,class_mode='categorical')

Found 18504 images belonging to 10 classes.
Found 4626 images belonging to 10 classes.


In [24]:
model = Sequential([
    Conv2D(32,(3,3),activation='relu',input_shape=(128,128,3)),
    MaxPool2D((2,2)),
    Conv2D(64,(3,3),activation='relu'),
    MaxPool2D((2,2)),
    Flatten(),
    Dense(128,activation='relu'),
    Dropout(0.3),
    Dense(len(train_data.class_indices),activation='sigmoid')
])
model.compile(optimizer='adam',loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

C:\Users\kprat\anaconda3\lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)                    │ (None, 126, 126, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_8 (MaxPooling2D)       │ (None, 63, 63, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_10 (Conv2D)                   │ (None, 61, 61, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_9 (MaxPooling2D)       │ (None, 30, 30, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_4 (Flatten)                  │ (None, 57600)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 128)                 │       7,372,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 10)                  │           1,290 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 7,393,610 (28.20 MB)

 Trainable params: 7,393,610 (28.20 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
early_stopping = EarlyStopping(monitor='val_loss',patience=2,restore_best_weights=True)
history = model.fit(train_data,validation_data=val_data,epochs=2,batch_size=batch_size,callbacks=[early_stopping])

Epoch 1/2
579/579 ━━━━━━━━━━━━━━━━━━━━ 129s 222ms/step - accuracy: 0.8542 - loss: 0.4211 - val_accuracy: 0.9200 - val_loss: 0.2284
Epoch 2/2
579/579 ━━━━━━━━━━━━━━━━━━━━ 123s 213ms/step - accuracy: 0.9243 - loss: 0.2227 - val_accuracy: 0.9423 - val_loss: 0.1632


In [30]:
loss,accuracy = model.evaluate(val_data)
print("Loss:",loss,"Accuracy:",accuracy)

145/145 ━━━━━━━━━━━━━━━━━━━━ 6s 44ms/step - accuracy: 0.9461 - loss: 0.1517
Loss: 0.16321276128292084 Accuracy: 0.9422827363014221


In [32]:
model.save("DL_LA3.h5")

In [41]:
model = tf.keras.models.load_model("DL_LA3.h5")
labels = [
    "Apple_Apple_scab",
    "Apple_Black_rot",
    "Apple_Cedar_apple_rust",
    "Apple_healthy",
    "Blueberry_healthy",
    "Cherry_(including_sour)_healthy",
    "Cherry_(including_sour)_Powdery_mildew",
    "Corn_(maize)_Cercospora_leaf_spot_Gray_leaf_spot",
    "Corn_(maize)_Common_rust_",
    "Corn_(maize)_healthy"
]

img = tf.keras.utils.load_img(r"C:\XPrathmesh\College Stuff\DeepLearning\LA3_Dataset\New Plant Diseases Dataset(Augmented)\test\PotatoHealthy2.JPG",target_size=image_size)
img = np.expand_dims(img_to_array(img)/255.0,axis=0)
print(labels[np.argmax(model.predict(img))])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
Apple_Black_rot
